# Dataset con timestamps

En esta notebook obtendremos un dataset en formato json que nos permitirá recrear todo el dataset con un script de python

Como recordatorio, el dataset está conformado por dos datasets diferentes. Uno de 1 hora, proveniente de segmentos de YouTube y otro de 3 horas, proveniente de las Narraciones Mayas de Campeche.

Dividiremos esta notebook en los siguientes pasos:

- Paso 0. Importar librerías y nuestro dataset de huggingface
- Paso 1. Para el dataset de 1h ---- "1h-raw-data.csv"
- Paso 2. Para el dataset de 3h. A partir de "3h-transcripts.csv" asociar las transcripciones de huggingface con los timestamps de este csv.
- Paso 3. Combinar ambos datasets y exportarlos en formato json.

## Paso 0

In [1]:
from huggingface_hub import notebook_login
from pathlib import Path
import json
import re
import pandas as pd
from datasets import Audio, Dataset, Features, Value, load_dataset

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"

RAW_DATA_1H = DATA_DIR / "1h-raw-data.csv"
RAW_DATA_3H = DATA_DIR / "3h-transcripts.csv"
SPK_META_PATH = DATA_DIR / "spk_metadata.csv"

# Puente utt_id (spk-based, HF) <-> filename (id crudo url-derivado)
BRIDGE_PATH = DATA_DIR / "dataset.csv"
# Fuentes ya curadas de spk_001-018 (YouTube + grabaciones propias):
# traen url / title / spanish / timestamps
SEGMENTS_JSON_IN = DATA_DIR / "corrected_segments_out.json"
# Salida: JSON unificado de fuentes (fuente de verdad para segmentar desde 0)
OUT_JSON = DATA_DIR / "source_segments.json"

# Ubicacion (relativa al root del proyecto) de los mp3 crudos INALI del 3h.
# Se guarda tal cual en la url para que LocalAudioProcessor la resuelva como paths.root / url.
INALI_MP3_URL_DIR = "assets/sources/narraciones_mayas_campeche/mp3"

REPO_ID = "mau-cr/mayan-voice"

In [2]:
notebook_login()

In [3]:
hf_ds = load_dataset(REPO_ID)

In [4]:
ds = hf_ds["train"]

In [5]:
ds

Dataset({
    features: ['audio', 'maya', 'utt_id', 'spk_id'],
    num_rows: 2535
})

## Paso 1

In [6]:
import pandas as pd

raw_1h_df = pd.read_csv(RAW_DATA_1H)
raw_1h_df

,utt_id,maya,spanish,spk_id,start,end
0,c5EgkTbau2o_0000,baach,chachalaca,spk_001,00:00:22.550,00:00:24.450
1,c5EgkTbau2o_0001,chiich,abuela,spk_001,00:00:26.450,00:00:28.250
2,c5EgkTbau2o_0002,ch'íich',pájaro,spk_001,00:00:29.750,00:00:31.350
3,c5EgkTbau2o_0003,ja',agua,spk_001,00:00:32.450,00:00:33.550
4,c5EgkTbau2o_0004,kool,milpa,spk_001,00:00:35.350,00:00:37.050
...,...,...,...,...,...,...
1303,amigo_de_camacho_grabacion_17_0000,mixba'al,De nada.,spk_018,00:00:00.000,00:00:01.050
1304,amigo_de_camacho_grabacion_18_0000,ka anaktech jump'éel jats'uts k'iin,Que tengas un bonito día.,spk_018,00:00:00.000,00:00:02.322
1305,amigo_de_camacho_grabacion_19_0000,ka xi'iktech utsil,Que te vaya bien.,spk_018,00:00:00.000,00:00:01.588
1306,amigo_de_camacho_grabacion_20_0000,je'el k-ilikba'e',Nos vemos.,spk_018,00:00:00.000,00:00:01.489


In [7]:
# Paso 1 - Enriquecimiento del 1h (spk_001-018: YouTube + grabaciones propias)
#
# raw_1h_df["utt_id"] es el id CRUDO url-derivado (ej. c5EgkTbau2o_0000), que coincide
# con la columna "filename" del puente dataset.csv. Aqui solo extraemos spanish + timestamps;
# el utt_id spk-based y la maya canonica se asocian en el Paso 3 via el puente + HF.
enrich_1h = (
    raw_1h_df
    .rename(columns={"utt_id": "filename"})
    .loc[:, ["filename", "spanish", "start", "end"]]
    .copy()
)
print("enrich_1h:", enrich_1h.shape)
enrich_1h.head()

enrich_1h: (1308, 4)


,filename,spanish,start,end
0,c5EgkTbau2o_0000,chachalaca,00:00:22.550,00:00:24.450
1,c5EgkTbau2o_0001,abuela,00:00:26.450,00:00:28.250
2,c5EgkTbau2o_0002,pájaro,00:00:29.750,00:00:31.350
3,c5EgkTbau2o_0003,agua,00:00:32.450,00:00:33.550
4,c5EgkTbau2o_0004,milpa,00:00:35.350,00:00:37.050


## Paso 2

In [8]:
raw_3h_df = pd.read_csv(RAW_DATA_3H)
raw_3h_df

,audio_file,segment_id,time_start,time_end,duration,transcription
0,01_Anatolio_Pech,1,0.000,14.624,14.624,le tzicbalob nu kaaba'e xta'cun bixunan xta'cu...
1,01_Anatolio_Pech,2,14.624,25.312,10.688,a ta' kun vi xonaano le xonaano' jun p'ee co'l...
2,01_Anatolio_Pech,3,25.312,38.496,13.184,u kaaba'e' ya'ax che' paalo meeque ya'ala' ti'...
3,01_Anatolio_Pech,4,38.496,49.280,10.784,caan yila' t nupital u yaabitale' quyoco ti ch...
4,01_Anatolio_Pech,5,49.280,61.568,12.288,pero leti' utucul beyo le can wenequele u yumi...
...,...,...,...,...,...,...
1222,15_Mario_Chan,223,2246.016,2260.640,14.624,ca oce a manpure ca ada administracion pero ti...
1223,15_Mario_Chan,224,2260.640,2268.128,7.488,como toon un laj espricado too unbe taan too n...
1224,15_Mario_Chan,225,2268.128,2277.696,9.568,i le cus kaakpachkobe ma tu pakobi as ma tu pa...
1225,15_Mario_Chan,226,2277.696,2283.392,5.696,ten shamantiouye camiu nada siaa rosquiibiziq


In [9]:
# Paso 2 - Enriquecimiento del 3h (narraciones INALI, spk_019-032)
#
# El 3h NO tiene traduccion al espanol -> spanish = "".
# Los timestamps vienen en segundos (relativos al recording completo) y los pasamos a HH:MM:SS.mmm.
# El filename crudo se reconstruye igual que en el puente: audio_file + "_" + segment_id (SIN zero-padding).

def seconds_to_hhmmss(x: float) -> str:
    total_ms = int(round(float(x) * 1000))
    h, total_ms = divmod(total_ms, 3_600_000)
    m, total_ms = divmod(total_ms, 60_000)
    s, ms = divmod(total_ms, 1000)
    return f"{h:02}:{m:02}:{s:02}.{ms:03}"

raw_3h_df["filename"] = raw_3h_df["audio_file"] + "_" + raw_3h_df["segment_id"].astype(str)

enrich_3h = pd.DataFrame({
    "filename": raw_3h_df["filename"],
    "spanish": "",
    "start": raw_3h_df["time_start"].map(seconds_to_hhmmss),
    "end": raw_3h_df["time_end"].map(seconds_to_hhmmss),
})
print("enrich_3h:", enrich_3h.shape)
enrich_3h.head()

enrich_3h: (1227, 4)


,filename,spanish,start,end
0,01_Anatolio_Pech_1,,00:00:00.000,00:00:14.624
1,01_Anatolio_Pech_2,,00:00:14.624,00:00:25.312
2,01_Anatolio_Pech_3,,00:00:25.312,00:00:38.496
3,01_Anatolio_Pech_4,,00:00:38.496,00:00:49.280
4,01_Anatolio_Pech_5,,00:00:49.280,00:01:01.568


## Paso 3 - Combinar y exportar

Reconciliamos todo tomando el **dataset de HF como columna vertebral** (aporta el `utt_id`
spk-based canonico, la `maya` y el `spk_id`) y le asociamos, por `utt_id`:

- **spanish + timestamps** (del Paso 1 y 2), via el puente `dataset.csv` (`utt_id` <-> `filename`).
- **url + title** de la fuente, desde un *registro de fuentes* (YouTube/propias del JSON curado; INALI = mp3 completo).

Finalmente agrupamos por fuente y exportamos el JSON unificado, que conserva el `utt_id`
spk-based en cada segmento para que el futuro script de segmentacion nombre los cortes igual
que el dataset publicado.

In [10]:
# ---- Paso 3.1 - Columna vertebral: dataset HF (utt_id spk-based, maya, spk_id) ----
# Accedemos por columna para NO decodificar el audio.
df = pd.DataFrame({
    "utt_id": ds["utt_id"],
    "maya":   ds["maya"],
    "spk_id": ds["spk_id"],
})

# ---- Paso 3.2 - Puente utt_id (spk) -> filename (id crudo) ----
bridge = pd.read_csv(BRIDGE_PATH)[["utt_id", "filename"]]
df = df.merge(bridge, on="utt_id", how="left")
assert df["filename"].notna().all(), "Hay utt_id del HF sin filename en el puente"

# ---- Paso 3.3 - Asociar spanish + timestamps (1h + 3h) por filename ----
enrich = pd.concat([enrich_1h, enrich_3h], ignore_index=True)
df = df.merge(enrich, on="filename", how="left")
df["spanish"] = df["spanish"].fillna("")

# ---- Paso 3.4 - Registro de fuentes: filename -> (url, title, metadata) ----
def video_id_from_url(url: str) -> str:
    """Mismo id que VideoAnnotation._get_url_id: YouTube -> id de 'v=', local -> carpeta_stem."""
    if url.startswith("http"):
        raw = url.split("v=")[1]
    else:
        p = Path(url)
        raw = f"{p.parent.name}_{p.stem}"
    return re.sub(r"[^A-Za-z0-9_-]+", "_", raw).strip("_")

with open(SEGMENTS_JSON_IN, encoding="utf-8") as f:
    curated = json.load(f)                      # spk_001-018 (YouTube + propias)

reg_rows = [
    {
        "source_key": video_id_from_url(s["url"]),
        "url": s["url"],
        "title": s["title"],
        "metadata": s.get("metadata", {}),
    }
    for s in curated
]
for af in sorted(raw_3h_df["audio_file"].unique()):   # spk_019-032 (INALI)
    reg_rows.append({
        "source_key": af,
        "url": f"{INALI_MP3_URL_DIR}/{af}.mp3",
        "title": af,
        "metadata": {},
    })
source_registry = pd.DataFrame(reg_rows).drop_duplicates("source_key")

# source_key = filename sin el sufijo "_<indice>"
df["source_key"] = df["filename"].str.replace(r"_\d+$", "", regex=True)
df = df.merge(source_registry, on="source_key", how="left")

print("df reconciliado:", df.shape)
df.head()

df reconciliado: (2535, 11)


,utt_id,maya,spk_id,filename,spanish,start,end,source_key,url,title,metadata
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000,chachalaca,00:00:22.550,00:00:24.450,c5EgkTbau2o,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca...",{'age': 'adult'}
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001,abuela,00:00:26.450,00:00:28.250,c5EgkTbau2o,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca...",{'age': 'adult'}
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002,pájaro,00:00:29.750,00:00:31.350,c5EgkTbau2o,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca...",{'age': 'adult'}
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003,agua,00:00:32.450,00:00:33.550,c5EgkTbau2o,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca...",{'age': 'adult'}
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004,milpa,00:00:35.350,00:00:37.050,c5EgkTbau2o,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca...",{'age': 'adult'}


In [11]:
# 3.5 Chequeos de integridad (no deberian saltar)
sin_url = df["url"].isna().sum()
sin_ts = (df["start"].isna() | df["end"].isna()).sum()
con_spanish = (df["spanish"] != "").sum()

es_3h = df["spk_id"].isin([f"spk_{i:03}" for i in range(19, 33)])
spanish_en_3h = ((df["spanish"] != "") & es_3h).sum()

print(f"filas totales      : {len(df)}")
print(f"sin url            : {sin_url}")
print(f"sin timestamps     : {sin_ts}")
print(f"con spanish (!= '') : {con_spanish}  (esperado: solo spk_001-018)")
print(f"spanish en 3h      : {spanish_en_3h}  (esperado: 0)")

assert sin_url == 0, "Hay segmentos sin fuente (url)"
assert sin_ts == 0, "Hay segmentos sin timestamps"
assert spanish_en_3h == 0, "El 3h no deberia tener spanish"

filas totales      : 2535
sin url            : 0
sin timestamps     : 0
con spanish (!= '') : 474  (esperado: solo spk_001-018)
spanish en 3h      : 0  (esperado: 0)


In [12]:
# 3.5b Anonimizar grabaciones propias + copiar audios crudos a raw_audios/
#
# Las grabaciones propias exponen el nombre real en url y title
# (ej. .../name/grabacion_1.wav, title "grabacion 1 name").
# Los reemplazamos por el spk_id. YouTube e INALI quedan intactos.
import shutil

RAW_AUDIOS_DIR = NOTEBOOK_DIR / "raw_audios"
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]

df["src_url"] = df["url"]  # ruta original (para copiar el audio crudo)

def _anon(row):
    u = row["src_url"]
    if "assets/sources/recordings/" not in u:
        return pd.Series({"url": row["url"], "title": row["title"]})   # youtube / INALI: sin cambios
    n = int(re.search(r"grabacion_(\d+)", u).group(1))
    spk = row["spk_id"].replace("_", "")           # spk_017 -> spk017
    anon = f"{spk}_{n:02d}"                          # spk017_01
    return pd.Series({"url": f"raw_audios/{anon}.wav", "title": anon})

df[["url", "title"]] = df.apply(_anon, axis=1)

# Copiar cada wav propio unico a raw_audios/ con el nombre anonimizado (copia, no destructiva)
RAW_AUDIOS_DIR.mkdir(exist_ok=True)
propias = (
    df[df["src_url"].str.contains("assets/sources/recordings/")]
    [["src_url", "url"]].drop_duplicates()
)
for src_url, dst_url in propias.itertuples(index=False):
    shutil.copy2(PROJECT_ROOT / src_url, NOTEBOOK_DIR / dst_url)

print(f"anonimizadas y copiadas: {len(propias)} grabaciones propias -> {RAW_AUDIOS_DIR}")

anonimizadas y copiadas: 100 grabaciones propias -> /home/maucr/Documentos/thesis-mayan-ai/notebooks/create_dataset/raw_audios


In [13]:
# 3.6 Construir el JSON unificado agrupado por fuente y exportar
#     Estructura identica a corrected_segments_out.json (url, title, segments, metadata),
#     pero con utt_id explicito por segmento.
df_sorted = df.sort_values(["source_key", "start"], kind="stable")

records = []
for (url, title), g in df_sorted.groupby(["url", "title"], sort=False):
    segments = [
        {
            "utt_id": r.utt_id,
            "maya": r.maya,
            "spanish": r.spanish,
            "start": r.start,
            "end": r.end,
            "spk_id": r.spk_id,
        }
        for r in g.itertuples(index=False)
    ]
    records.append({
        "url": url,
        "title": title,
        "segments": segments,
        "metadata": g["metadata"].iloc[0],
    })

OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

total_seg = sum(len(r["segments"]) for r in records)
print(f"Escrito {OUT_JSON}")
print(f"fuentes: {len(records)} | segmentos: {total_seg}")

Escrito /home/maucr/Documentos/thesis-mayan-ai/notebooks/create_dataset/data/source_segments.json
fuentes: 159 | segmentos: 2535


In [16]:
# 3.7 Validacion round-trip con el pipeline (SpokenDictionaryManifest + VideoAnnotation)
#     Confirma que el JSON carga y que conserva los utt_id spk-based (gracias a Parte B).
from kinai.data_collection.spoken_dictionary_manifest import SpokenDictionaryManifest

manifest = SpokenDictionaryManifest(OUT_JSON)
rt = manifest.get_segment_file_to_df()

assert len(rt) == len(df), "El round-trip cambio el numero de segmentos"
assert set(rt["utt_id"]) == set(df["utt_id"]), "El round-trip no preservo los utt_id"
assert rt["utt_id"].str.startswith("spk_").all(), "Se perdieron los utt_id spk-based"
print("OK: round-trip conserva", len(rt), "segmentos con utt_id spk-based")
rt.head()

OK: round-trip conserva 2535 segmentos con utt_id spk-based


,utt_id,maya,spanish,spk_id,start,end
0,spk_019_utt_0001,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,,spk_019,00:00:00.000,00:00:14.624
1,spk_019_utt_0002,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,,spk_019,00:00:14.624,00:00:25.312
2,spk_019_utt_0003,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,,spk_019,00:00:25.312,00:00:38.496
3,spk_019_utt_0004,káan u yila tu bin tu taal u yáa'biltale ku yo...,,spk_019,00:00:38.496,00:00:49.280
4,spk_019_utt_0005,pero leti'e u tuukul beyo le káan weenek le u ...,,spk_019,00:00:49.280,00:01:01.568


In [15]:
# 3.8 OPCIONAL - Actualizar el dataset HF con las columnas nuevas (spanish, start, end, url)
#
# add_column preserva el feature Audio existente (no re-castea el audio).
# Descomenta para publicar. Requiere notebook_login() con token de escritura.

# ds_enriched = ds
# for col in ["spanish", "start", "end", "url"]:
#     m = dict(zip(df["utt_id"], df[col]))
#     ds_enriched = ds_enriched.add_column(col, [m.get(u, "") for u in ds["utt_id"]])
# ds_enriched.push_to_hub(REPO_ID, private=True)
# ds_enriched